<a href="https://colab.research.google.com/github/annakalinina18/star-fle/blob/main/corpus_to_inception/sort_by_occurrences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import re

df = pd.read_excel("corpus_with_projected_mwes.xlsx")

rows = []

for _, row in df.iterrows():

    projected = str(row["projected_mwes"])

    # вытаскиваем ВСЕ expression
    expressions = re.findall(
        r'"expression"\s*:\s*"([^"]+)"',
        projected
    )

    for expr in expressions:

        occ = f"[{row['niveau CECRL']}] {row['sentence']}"

        rows.append({
            "expression": expr,
            "niveau_CECRL": row["niveau CECRL"],
            "occurrence": occ
        })

long_df = pd.DataFrame(rows)

# группировка
grouped = (
    long_df
    .groupby("expression")
    .agg(
        niveau_CECRL_supposé=("niveau_CECRL", "min"),
        occurrence_count=("occurrence", "count"),
        occurrences=("occurrence", list)
    )
    .reset_index()
)

# разворачиваем occ_1, occ_2...
max_occ = grouped["occurrences"].apply(len).max()

for i in range(max_occ):
    grouped[f"occ_{i+1}"] = grouped["occurrences"].apply(
        lambda x: x[i] if i < len(x) else None
    )

grouped.drop(columns=["occurrences"], inplace=True)

# пустые колонки как во второй таблице
grouped["annotation_resolved"] = None
grouped["annotation_summary_vote"] = None

# порядок колонок
cols = (
    ["expression",
     "annotation_resolved",
     "annotation_summary_vote",
     "niveau_CECRL_supposé",
     "occurrence_count"]
    +
    [c for c in grouped.columns if c.startswith("occ_")]
)

grouped = grouped[cols]

# сортировка
grouped = grouped.sort_values("expression")

grouped.to_excel(
    "mwe_occurrences_like_second_table.xlsx",
    index=False
)

print("Готово")

/tmp/ipykernel_19291/2479481790.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped[f"occ_{i+1}"] = grouped["occurrences"].apply(
/tmp/ipykernel_19291/2479481790.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  grouped[f"occ_{i+1}"] = grouped["occurrences"].apply(
/tmp/ipykernel_19291/2479481790.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. 

Готово
